# 文脈自由文法を書いて構文解析する (NLTK)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/B_implementations/cfg_parse.ipynb)

NLP クックブック B-13 ／ 仕組みを理解するための実装

自分で書いた小さな文法で文を解析し、構文木と構造的曖昧性を確かめたい (授業の演習向け)。

In [1]:
import nltk

grammar = nltk.CFG.fromstring("""
    S   -> NP VP
    NP  -> Det N | Det N PP | 'I'
    VP  -> V NP | VP PP
    PP  -> P NP
    Det -> 'an' | 'my' | 'the'
    N   -> 'elephant' | 'pajamas' | 'telescope' | 'man'
    V   -> 'shot' | 'saw'
    P   -> 'in' | 'with'
""")
parser = nltk.ChartParser(grammar)

sentence = "I shot an elephant in my pajamas".split()
trees = list(parser.parse(sentence))
print(len(trees), "通りの解析")
for tree in trees:
    tree.pretty_print()

2 通りの解析
     S                                       
  ___|______________                          
 |                  VP                       
 |         _________|__________               
 |        VP                   PP            
 |    ____|___              ___|___           
 |   |        NP           |       NP        
 |   |     ___|_____       |    ___|_____     
 NP  V   Det        N      P  Det        N   
 |   |    |         |      |   |         |    
 I  shot  an     elephant  in  my     pajamas

     S                                   
  ___|__________                          
 |              VP                       
 |    __________|______                   
 |   |                 NP                
 |   |     ____________|___               
 |   |    |     |          PP            
 |   |    |     |       ___|___           
 |   |    |     |      |       NP        
 |   |    |     |      |    ___|_____     
 NP  V   Det    N      P  Det        N   
 |   |    |  

In [2]:
# 確率文脈自由文法 (PCFG) で最も確からしい木を選ぶ
pcfg = nltk.PCFG.fromstring("""
    S   -> NP VP [1.0]
    NP  -> Det N [0.5] | Det N PP [0.2] | 'I' [0.3]
    VP  -> V NP [0.6] | VP PP [0.4]
    PP  -> P NP [1.0]
    Det -> 'an' [0.5] | 'my' [0.5]
    N   -> 'elephant' [0.5] | 'pajamas' [0.5]
    V   -> 'shot' [1.0]
    P   -> 'in' [1.0]
""")
best = next(nltk.ViterbiParser(pcfg).parse(sentence))
print(best.prob())
best.pretty_print()

0.001125
     S                                   
  ___|__________                          
 |              VP                       
 |    __________|______                   
 |   |                 NP                
 |   |     ____________|___               
 |   |    |     |          PP            
 |   |    |     |       ___|___           
 |   |    |     |      |       NP        
 |   |    |     |      |    ___|_____     
 NP  V   Det    N      P  Det        N   
 |   |    |     |      |   |         |    
 I  shot  an elephant  in  my     pajamas



- 「パジャマを着た象を撃った / パジャマ姿で象を撃った」という前置詞句付加の曖昧性の古典的な例。